In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
cafe_p = pd.read_csv('/home/panda2bat/Avivorous_bat/output/13_evolution-family_analysis/CAFE2/double/Base_branch_probabilities.tab', sep = '\t')
cafe_c = pd.read_csv('/home/panda2bat/Avivorous_bat/output/13_evolution-family_analysis/CAFE2/double/Base_change.tab', sep = '\t')
cafe_n = pd.read_csv('/home/panda2bat/Avivorous_bat/output/13_evolution-family_analysis/CAFE2/double/Base_count.tab', sep = '\t')

In [3]:
def change_columns(df):
    columns_lst = list()
    for i in df.columns:
        i = re.sub(r'[#<>]', '', i)
        if i == 'FamilyID':
            i = i
        else:
            if not i.isdigit():
                i = re.sub(r'[0-9]*', '', i)
            else:
                i = 'n_' + i
        columns_lst.append(i)
    df.columns = columns_lst
    return(df)

In [4]:
cafe_p = change_columns(cafe_p)
cafe_c = change_columns(cafe_c)
cafe_n = change_columns(cafe_n)

In [33]:
def parse_species(species, cafe_p, cafe_c, cafe_n):
    sig = cafe_p[cafe_p.loc[:, species] < 0.01].loc[:, ['FamilyID', species]]
    cha = cafe_c[cafe_c.FamilyID.isin(sig.FamilyID)].loc[:, ['FamilyID', species]]
    num = cafe_n[cafe_n.FamilyID.isin(sig.FamilyID)].loc[:, ['FamilyID', species]]
    merge = pd.merge(sig, cha, on = 'FamilyID', how = 'left', suffixes=['_p', '_c'])
    merge = pd.merge(merge, num, on = 'FamilyID', how = 'left')
    merge = merge.rename(columns = {species:f'{species}_n'})
    increase = len(cha[cha.loc[:, species] > 0])
    decrease = len(cha[cha.loc[:, species] < 0])
    return(increase, decrease, merge)
    

In [26]:
increase, decrease, num = parse_species('EptFus', cafe_p, cafe_c, cafe_n)

In [27]:
num[num.loc[:,'EptFus_n'] < 0]

,FamilyID,EptFus_p,EptFus_c,EptFus_n


In [34]:
df_tmp = {'species':[],'increase':[], 'decrease':[], 'merge':[]}
for i in list(cafe_c.columns)[1:]:
    increase, decrease, merge = parse_species(i, cafe_p, cafe_c, cafe_n)
    df_tmp['species'].append(i)
    df_tmp['increase'].append(increase)
    df_tmp['decrease'].append(decrease)
    df_tmp['merge'].append(merge)
final = pd.DataFrame(df_tmp)

,species,increase,decrease,merge
0,EptFus,378,0,FamilyID EptFus_p EptFus_c Ep...
1,IaIo,303,139,FamilyID IaIo_p IaIo_c IaIo...
2,NycAvi,164,53,FamilyID NycAvi_p NycAvi_c Ny...
3,PipKuh,333,11,FamilyID PipKuh_p PipKuh_c Pi...
4,MyoBra,342,22,FamilyID MyoBra_p MyoBra_c My...
5,MyoLuc,386,9,FamilyID MyoLuc_p MyoLuc_c My...
6,MyoDav,182,48,FamilyID MyoDav_p MyoDav_c My...
7,MyoMyo,384,42,FamilyID MyoMyo_p MyoMyo_c My...
8,n_8,57,2,FamilyID n_8_p n_8_c n_8_n ...
9,n_9,35,8,FamilyID n_9_p n_9_c n_9_n ...
